# NutritionLLM change check

A half-hour Kaggle run that checks a code change on real data before a full run: the quick try with
the current `train.py` (batches, 16-bit numbers), both tests at once on the two GPUs, and a sweep of
the repeat penalty on the 40 most crowded lab plates plus 15 typical ones, using the adapter from the
2026-10-06 run. That adapter comes from the Kaggle dataset `nutritionllm-full-usda` named in
`kernel-metadata.json`; make it once from `runs/full_usda` with `kaggle datasets create -p <folder>`
(the folder holds the adapter files and a `dataset-metadata.json`), and refresh it with
`kaggle datasets version` after a better run. Look for the loss and peak memory of the quick try, two
readable test tables, and `results/penalty_<value>.json` for each penalty.

From the project folder: `kaggle kernels push -p notebooks/change_check`, then
`kaggle kernels output <your-username>/nutritionllm-change-check -p results`.

In [ ]:
# Get the code; /kaggle/temp has room for the 16 GB of data, and only /kaggle/working is kept
!mkdir -p /kaggle/temp
%cd /kaggle/temp
!git clone https://github.com/AadiKenwar1/NutritionLLM.git
%cd NutritionLLM

In [ ]:
# Install the tested library versions; Kaggle's old torchao makes peft refuse to load
!pip install -q -r requirements.txt
!pip uninstall -y torchao

In [ ]:
# Download the four datasets and build the training and test files
!python download.py
!python process.py

In [ ]:
# Quick try of the training code, then both tests at once, one per GPU
!python train.py --limit 40 --out runs/try
!(nvidia-smi -L | grep -q "GPU 1" && export CUDA_VISIBLE_DEVICES=1; python test.py --model runs/try --data data/processed/test_lab.jsonl --limit 5) > results/try_test_lab.log 2>&1 & CUDA_VISIBLE_DEVICES=0 python test.py --model runs/try --data data/processed/test_real_world.jsonl --limit 5; wait; tail -n 25 results/try_test_lab.log

In [ ]:
# Safety stop: both tests must have written a report
import os
assert os.path.exists("results/try_test_real_world.json") and os.path.exists("results/try_test_lab.json"), "The quick try failed."

In [ ]:
# Crowded lab plates, where the last run looped into endless lists, plus typical plates to catch side effects
import json
rows = [json.loads(l) for l in open("data/processed/test_lab.jsonl")]
crowded = sorted(rows, key=lambda r: -len(r["answer"]["items"]))[:40]
typical = [r for r in rows if 3 <= len(r["answer"]["items"]) <= 5][:15]
with open("data/processed/test_lab_subset.jsonl", "w") as f:
    for r in crowded + typical:
        print(json.dumps(r), file=f)

In [ ]:
# The same 55 plates at three repeat penalties with the Oct 6 adapter -> results/penalty_<value>.json
import os, subprocess
for p in ("1.05", "1.2", "1.3"):
    cmd = f"python test.py --model /kaggle/input/nutritionllm-full-usda --data data/processed/test_lab_subset.jsonl --limit 0 --penalty {p}"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"=== penalty {p}")
    print(r.stdout[-1200:], r.stderr[-600:] if r.returncode else "")
    for ext in ("json", "jsonl"):
        os.replace(f"results/nutritionllm-full-usda_test_lab_subset.{ext}", f"results/penalty_{p}.{ext}")

In [ ]:
# Pack the results; only /kaggle/working survives the run
!zip -r /kaggle/working/output.zip results